# Tutorial: Config Interfaces and Scenario Resolution

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Model maintainers editing run/scenario config contracts.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Load and inspect the full run config object.
- Understand variant and dimension override resolution per slice.
- Inspect temporal forms and report-phase enforcement behavior.


## Outline

1. Load run config and variants
2. Inspect baseline parameter blocks
3. Resolve slice-level overrides
4. Inspect temporal forms in scenario files
5. Validate report-phase no-pre-report modifications behavior


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Load config model


In [ ]:
if load_run_config is None:
    raise RuntimeError("crm_model not importable")
cfg = load_run_config(CONFIG_PATH)
print("Run:", cfg.name)
print("Variants:", len(cfg.variants))


## Step 2: Inspect baseline interface blocks


In [ ]:
for block in ["sd_parameters", "mfa_parameters", "strategy", "transition_policy", "demand_transformation"]:
    data = getattr(cfg, block, None)
    if data is None:
        print(block, "missing")
        continue
    if hasattr(data, "model_dump"):
        d = data.model_dump(exclude_none=True, exclude_unset=True)
    elif isinstance(data, dict):
        d = data
    else:
        d = {}
    print(f"{block}: {len(d)} keys")
    for k in list(d.keys())[:8]:
        print(" -", k)


## Step 3: Resolve one slice (material-region) for a variant


In [ ]:
from crm_model.scenarios import resolve_variant_slice_overrides

variant = EXAMPLE_VARIANT
material = "nickel"
region = "EU27"
sl = resolve_variant_slice_overrides(cfg=cfg, variant_name=variant, material=material, region=region)
print("Variant:", variant, "Slice:", material, region)
for block in ["sd_parameters", "mfa_parameters", "strategy", "transition_policy", "demand_transformation", "shocks"]:
    keys = list(sl.get(block, {}).keys())
    print(f"{block}: {len(keys)} keys")
    if keys:
        print("  sample:", keys[:6])


## Step 4: Inspect temporal value forms in scenario YAML


In [ ]:
import yaml

scenario_file = REPO / "configs" / "scenarios" / "mvp" / "circularity_push.yml"
raw = yaml.safe_load(scenario_file.read_text(encoding="utf-8"))
print("Scenario:", raw.get("name"))

for block in ["sd_parameters", "mfa_parameters", "strategy", "demand_transformation"]:
    b = raw.get(block, {}) or {}
    if not isinstance(b, dict):
        continue
    print(f"\n{block} temporal-form scan")
    for k, v in b.items():
        if isinstance(v, dict) and "exogenous_ramp" in v:
            print(" -", k, "exogenous_ramp")
        elif isinstance(v, dict) and "start_year" in v and "value" in v:
            print(" -", k, "year_gate")
        elif isinstance(v, list):
            print(" -", k, "timeseries")


## Step 5: Demonstrate report-phase temporal enforcement


In [ ]:
from crm_model.cli import _enforce_reporting_phase_for_variant_slice

years = list(range(cfg.time.start_year, cfg.time.end_year + 1))
report_start = int(cfg.time.report_start_year)

demo_slice = {
    "sd_parameters": {"capacity_expansion_gain": {"start_year": 2010, "value": 0.5}},
    "mfa_parameters": {},
    "strategy": {},
    "transition_policy": {},
    "demand_transformation": {},
    "shocks": {"demand_surge": {"start_year": 2010, "duration_years": 10, "multiplier": 1.2}},
}

out = _enforce_reporting_phase_for_variant_slice(
    variant_slice=demo_slice,
    years=years,
    report_start_year=report_start,
    sd_base={"capacity_expansion_gain": 0.26},
    mfa_base={},
    strategy_base={},
    transition_policy_base={},
    demand_transformation_base={},
    shocks_base={},
)
print(out["sd_parameters"]["capacity_expansion_gain"])
print(out["shocks"]["demand_surge"])


## Pitfalls

- Assuming scenario YAML values are already resolved per slice.
- Forgetting that report-phase enforcement can clip/shift temporal activations.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
